Dataset Link: https://www.kaggle.com/datasets/ninadaithal/imagesoasis

This dataset requires lot of preprocessing because a single patients MRI Scan has 61 images. So we need to first separate all the distinct patient before we proceed

In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import Input, Sequential, layers, mixed_precision  # type: ignore

mixed_precision.set_global_policy("mixed_float16")

In [ ]:
print("Compute dtype: %s" % mixed_precision.global_policy().compute_dtype)
print("Variable dtype: %s" % mixed_precision.global_policy().variable_dtype)

In [ ]:
data_dir = "./Data/MRI_Dataset"

filenames = []
labels = []

In [ ]:
class_names = sorted(
    [f for f in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, f))]
)
class_index = {class_names: idx for idx, class_names in enumerate(class_names)}

print(class_index)  # Class mapping

In [ ]:
for class_name in class_names:
    class_path = os.path.join(data_dir, class_name)
    current_label = class_index[class_name]
    for file in os.listdir(class_path):
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            file_path = os.path.join(class_path, file)
            filenames.append(file_path)
            labels.append(current_label)


filenames = np.array(filenames)
labels = np.array(labels)
groups = np.array([os.path.basename(f)[:9] for f in filenames])

From the output of the code block below, we can see that class label 1: Moderate Dementia, we only have 488 images. One patient has multiple scans in this dataset so on an average each patient has 244 images. So Moderate Dementia only has data of 2 patients, which is useless for training.

In [ ]:
values, counts = np.unique(labels, return_counts=True)
print(values, counts)

print(len(np.unique(groups)))

So, I am going to merge moderate dementia with mild dementia. Because we are removing moderate dementia it's label 1 is being changed to 0 and label 2 and label 3 needs to be renamed as label 1 and label 2 respectively

In [ ]:
print("Original label distribution:", counts)

labels[labels == 1] = 0
labels[labels == 2] = 1
labels[labels == 3] = 2

new_values, new_counts = np.unique(labels, return_counts=True)
print("\n--- After Merging and Reindexing ---")
print(f"New unique labels: {new_values}")
print(f"New sample counts: {new_counts}")

Splitting the data into train, validation and test split

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=8)

train_val_index, test_index = next(gss.split(filenames, labels, groups=groups))
x, x_test = filenames[train_val_index], filenames[test_index]
y, y_test = labels[train_val_index], labels[test_index]
groups_train_val = groups[train_val_index]

train_index, val_index = next(gss.split(x, y, groups=groups_train_val))
x_train, x_val = x[train_index], x[val_index]
y_train, y_val = y[train_index], y[val_index]

To verify if GroupShuffleSplit actually split the data correctly

In [ ]:
print(x_train[500], y_train[500])

In [ ]:
train_patients = set(groups_train_val[train_index])
val_patients = set(groups_train_val[val_index])
test_patients = set(groups[test_index])

intersection_trval = train_patients.intersection(val_patients)
intersection = set(groups[train_val_index]).intersection(test_patients)

print(f"Unique patients in Training Set: {len(train_patients)}")
print(f"Unique patients in Validation Set: {len(val_patients)}")
print(f"Unique patients in Testing Set: {len(test_patients)}")
print(
    f"Number of patients leaking into Train and Validation sets: {len(intersection_trval)}"
)
print(f"Number of patients leaking into Train_Val and Test: {len(intersection)}")

In [ ]:
def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=1)
    img = tf.image.resize(img, [128, 128])
    img = img / 255.0

    # label = tf.one_hot(label, depth=3)

    return img, label


train_dataset = tf.data.Dataset.from_tensor_slices(
    (x_train, y_train)
)  # Here x_train is the path of the image, y_train is the label4
train_dataset = train_dataset.map(load_and_preprocess)
train_dataset = train_dataset.shuffle(buffer_size=1000)
train_dataset = train_dataset.batch(64).prefetch(tf.data.AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((x_test, y_test))
test_dataset = (
    test_dataset.map(load_and_preprocess).batch(64).prefetch(tf.data.AUTOTUNE)
)


val_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val))
val_dataset = val_dataset.map(load_and_preprocess).batch(64).prefetch(tf.data.AUTOTUNE)

echo "LD_LIBRARY_PATH=$(find $PWD/.venv -type d -name "lib" -path "*/nvidia/*" | paste -sd : -)" > .env

In [ ]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices("GPU")))

Computing class weights to mitigate the effects of class imbalance

In [ ]:
class_weight_array = compute_class_weight(
    class_weight="balanced", classes=np.unique(y_train), y=y_train
)

print(class_weight_array)

class_weight_dict = dict(enumerate(class_weight_array))
print(class_weight_dict)

We have to be very careful during data augmentation here, because MRI Scans can be augmented is such way that would make it anatomically impossible. We keep padding=same because we want to look all the edges too. We use Flatten instead of Global Pooling layer because we want to preserve spatial relation 

In [ ]:
from numpy import float32

model = Sequential()
model.add(Input(shape=(128, 128, 1)))
model.add(layers.RandomFlip("horizontal"))
model.add(layers.RandomRotation(factor=0.05))
model.add(layers.RandomZoom(height_factor=0.05, width_factor=0.05))

model.add(layers.Conv2D(32, (3, 3), activation="relu", padding="same"))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D(pool_size=(2, 2)))
model.add(layers.Dropout(0.2))

model.add(layers.Conv2D(128, (3, 3), activation="relu", padding="same"))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D(pool_size=(2, 2)))
model.add(layers.Dropout(0.2))

model.add(layers.Conv2D(256, (3, 3), activation="relu", padding="same"))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D(pool_size=(2, 2)))
model.add(layers.Dropout(0.2))

model.add(layers.Flatten())

model.add(layers.Dense(128, activation="relu"))
model.add(layers.BatchNormalization())
model.add(layers.Dropout(0.5))

model.add(layers.Dense(3, activation="softmax", dtype=float32))
model.summary()

Cannot use categorical_crossentropy as labels are formatted as one dimensional np array. We need to use sparse_categorical_crossentropy. Drawback here is that we lose the soft label from categorical_crossentropy (Eg: 70% Dog, 20% Cat)

Edit: To use metrics like AUC, Precision and Recall, I need to use one hot encoding

Edit2: Apparently using one hot encoding with class weights absolutely wrecks the metrics, so back to using sparse_categorical_crossentropy

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    # metrics=[keras.metrics.AUC(), keras.metrics.Precision(), keras.metrics.Recall()],
    metrics=["sparse_categorical_accuracy"],
)

In [ ]:
history = model.fit(
    x=train_dataset,
    epochs=30,
    class_weight=class_weight_dict,
    validation_data=val_dataset,
)

In [ ]:
# Access the raw dictionary
metrics_dict = history.history

# Print the full list of your training and validation AUC across all epochs
print("Training AUC history:", metrics_dict["auc"])
print("Validation AUC history:", metrics_dict["val_auc"])